In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer
from transformers.utils import logging
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
)


def binary_metrics(y_true, y_pred, y_score=None):
    """Метрики с акцентом на редкий положительный класс Target=1."""
    result = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_1": precision_score(y_true, y_pred, zero_division=0),
        "Recall_1": recall_score(y_true, y_pred, zero_division=0),
        "F1_1": f1_score(y_true, y_pred, zero_division=0),
        "Balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }
    if y_score is not None:
        result["ROC_AUC"] = roc_auc_score(y_true, y_score)
    return result


def print_metrics(name, metrics):
    print(name)
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

1

In [ ]:
train = pd.read_csv("train.csv")

print("Исходный размер:", train.shape)

print("\nКоличество пропусков:")
print(train.isna().sum())

duplicates_count = train.duplicated(
    subset=["Prompt", "Answer", "Target"]
).sum()

print("\nКоличество дубликатов:", duplicates_count)

train = train.drop_duplicates(
    subset=["Prompt", "Answer", "Target"],
    keep="first"
)

train = train.dropna(
    subset=["Prompt", "Answer", "Target"]
)

print("\nРазмер после очистки:", train.shape)


train = train.reset_index(drop=True)

2

In [ ]:
class_counts = train["Target"].value_counts().sort_index()
class_share = train["Target"].value_counts(normalize=True).sort_index()

print("Количество объектов по классам:")
print(class_counts)
print("\nДоли классов:")
print(class_share)

class_counts.plot(kind="bar")
plt.xlabel("Target")
plt.ylabel("Количество")
plt.title("Распределение классов")
plt.show()

minority_class = class_counts.idxmin()
minority_share = class_share.loc[minority_class]
imbalance_ratio = class_counts.max() / class_counts.min()
print(
    f"Интерпретация: класс {minority_class} является редким и составляет "
    f"{minority_share:.2%} выборки. Соотношение размеров классов ≈ {imbalance_ratio:.1f}:1. "
    "Поэтому одной Accuracy недостаточно: дальше дополнительно используем precision, recall, F1, "
    "balanced accuracy и ROC-AUC."
)

In [ ]:
train["Prompt_len"] = train["Prompt"].str.len()
train["Answer_len"] = train["Answer"].str.len()

plt.figure(figsize=(12, 6))
plt.hist(train["Prompt_len"], bins=50, alpha=0.7, label="Prompt")
plt.hist(train["Answer_len"], bins=50, alpha=0.7, label="Answer")
plt.xlabel("Длина текста в символах")
plt.ylabel("Количество текстов")
plt.title("Распределение длин Prompt и Answer")
plt.legend()
plt.show()

prompt_median = train["Prompt_len"].median()
answer_median = train["Answer_len"].median()
prompt_p95 = train["Prompt_len"].quantile(0.95)
answer_p95 = train["Answer_len"].quantile(0.95)
print(
    f"Интерпретация: медианная длина Prompt = {prompt_median:.0f} символов, "
    f"Answer = {answer_median:.0f}. 95-й перцентиль: Prompt = {prompt_p95:.0f}, "
    f"Answer = {answer_p95:.0f}. Длинный правый хвост показывает наличие сравнительно редких "
    "очень длинных текстов, поэтому среднее значение здесь менее устойчиво, чем медиана."
)

In [5]:
pd.set_option("display.max_colwidth", None)


# !!! В А Ж Н О !!!
''' я убрал ограничение на вывод текста,
чтобы просмотреть примеры длинных и коротких ответов модели '''
''' надо использовать clear cell output
для того чтобы не было проблемы с прокруткой страницы с моим кодом'''

' надо использовать clear cell output\nдля того чтобы не было проблемы с прокруткой страницы с моим кодом'

In [6]:
train.loc[train["Id"] == 19947, "Target"] = 1
train.loc[train["Id"] == 22155, "Target"] = 0
train.loc[train["Id"] == 4132, "Target"] = 0

print(train.nsmallest(10, "Answer_len")[
    ["Id", "Prompt", "Answer", "Answer_len", "Target"]
])
#здесь я указываю id а не индексы
#19947 (a: 5) модель просят вывести букву ответа (a), а модель вывела содержание буквы ответа (5)
#значит то-что target = 0 это неправильно, заменил на target = 1
#22155 модель выводит букву ответа как её и ожидают, буква (c) есть в вариантах ответа
#значит то-что target = 1 это неправильно, заменил на target = 0
#4132 модель просят вывести описательное предложение, а модель выводит точку
#значит то-что target = 1 это неправильно, заменил на target = 0
#1046 модель не обьяснила ответ, хотя ответ 0 это один из возможных ответов даже если неправильный
#оставил как есть, сложный кейс, модель всё таки ничего не выдумала



        Id  \
46    1139   
109  19947   
225  12973   
293  22155   
295   4132   
438  24658   
465  11250   
492  10498   
529   1046   
544  26521   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [7]:
for _, row in train.nlargest(10, "Answer_len").iterrows():
    print("Id:", row["Id"])
    print("Prompt:", row["Prompt"])
    print("Answer:", row["Answer"])
    print("Answer_len:", row["Answer_len"])
    print("Target:", row["Target"])
    print("-" * 100)

Id: 5435
Prompt: [INST] You are an AI assistant. You will be given a task. You must generate a detailed and long answer. [/INST]
        What is the version of the following sentence with correct punctuation?

cumshot compilation group squirt compilation
Answer: cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilation cumshot compilati

In [8]:
#25512 ответ выходит за рамки опций
train.loc[train["Id"] == 25512, "Target"] = 1
#16857 ответ не удовлетворяет запросу
train.loc[train["Id"] == 16857, "Target"] = 1


3

In [ ]:
logging.set_verbosity_error()
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train["Prompt_tokens"] = train["Prompt"].apply(tokenizer.tokenize)
train["Answer_tokens"] = train["Answer"].apply(tokenizer.tokenize)

all_tokens = pd.concat([
    train["Prompt_tokens"].explode(),
    train["Answer_tokens"].explode()
]).dropna()

token_counts = all_tokens.value_counts()
top_20 = token_counts.head(20)
print(top_20)

plt.figure(figsize=(12, 6))
plt.bar(top_20.index, top_20.values)
plt.xlabel("Токен")
plt.ylabel("Частота")
plt.title("Топ-20 самых частых токенов")
plt.xticks(rotation=45)
plt.show()

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="black"
).generate_from_frequencies(top_20.to_dict())

plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Облако топ-20 самых частых токенов")
plt.show()

print(
    "Интерпретация: до очистки среди наиболее частых токенов ожидаемо много служебных слов. "
    "Они хорошо описывают частотность языка, но обычно несут мало информации о целевом классе. "
    "Следующий шаг удаляет английские stop-words и WordPiece-фрагменты."
)

In [ ]:
import nltk
from nltk.corpus import stopwords


try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    stop_words = set(stopwords.words("english"))


def clean_tokens(tokens):
    return [
        token
        for token in tokens
        if token not in stop_words
        and token.isalpha()  
    ]


train["Prompt_tokens_clean"] = train["Prompt_tokens"].apply(clean_tokens)
train["Answer_tokens_clean"] = train["Answer_tokens"].apply(clean_tokens)

all_clean_tokens = pd.concat([
    train["Prompt_tokens_clean"].explode(),
    train["Answer_tokens_clean"].explode()
]).dropna()

clean_token_counts = all_clean_tokens.value_counts()
top_20_clean = clean_token_counts.head(20)
print(top_20_clean)

plt.figure(figsize=(12, 6))
plt.bar(top_20_clean.index, top_20_clean.values)
plt.xlabel("Токен")
plt.ylabel("Частота")
plt.title("Топ-20 токенов после очистки")
plt.xticks(rotation=45)
plt.show()

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="black"
).generate_from_frequencies(top_20_clean.to_dict())

plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Облако слов после очистки")
plt.show()

removed_share = 1 - len(all_clean_tokens) / len(all_tokens)
print(
    f"Интерпретация: после удаления NLTK stop-words и неалфавитных/WordPiece-токенов "
    f"удалено примерно {removed_share:.1%} токенов. В частотном топе теперь больше содержательных слов, "
    "что потенциально уменьшает шум для линейной модели."
)

4

In [ ]:

y = train["Target"]
train_idx, test_idx = train_test_split(
    train.index,
    test_size=0.2,
    random_state=34,
    stratify=y
)

x_train_text = (train.loc[train_idx, "Prompt"] + " " + train.loc[train_idx, "Answer"])
x_test_text = (train.loc[test_idx, "Prompt"] + " " + train.loc[test_idx, "Answer"])
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

vectorizer = TfidfVectorizer()

_t0 = time.perf_counter()
x_train_vec = vectorizer.fit_transform(x_train_text)
baseline_vectorize_train_time = time.perf_counter() - _t0

_t0 = time.perf_counter()
x_test_vec = vectorizer.transform(x_test_text)
baseline_vectorize_test_time = time.perf_counter() - _t0

print("Размер TF-IDF матрицы train:", x_train_vec.shape)
print("Размер TF-IDF матрицы test:", x_test_vec.shape)

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

_t0 = time.perf_counter()
model.fit(x_train_vec, y_train)
baseline_model_train_time = time.perf_counter() - _t0

_t0 = time.perf_counter()
y_pred = model.predict(x_test_vec)
y_score = model.predict_proba(x_test_vec)[:, 1]
baseline_model_inference_time = time.perf_counter() - _t0

baseline_metrics = binary_metrics(y_test, y_pred, y_score)
baseline_train_time = baseline_vectorize_train_time + baseline_model_train_time
baseline_inference_time = baseline_vectorize_test_time + baseline_model_inference_time

print_metrics("Шаг 4 — TF-IDF + Logistic Regression на исходных текстах", baseline_metrics)
print(f"Train time (vectorization + fit): {baseline_train_time:.4f} s")
print(f"Inference time (transform + predict): {baseline_inference_time:.4f} s")

baseline_result = {
    "Model": "TF-IDF + LR (raw)",
    **baseline_metrics,
    "Train_time_s": baseline_train_time,
    "Inference_time_s": baseline_inference_time,
}

print(
    "\nИнтерпретация: при дисбалансе классов ориентируемся прежде всего на Recall_1/F1_1 и "
    "Balanced_accuracy, а не на Accuracy. ROC-AUC показывает качество ранжирования независимо "
    "от фиксированного порога 0.5."
)

In [ ]:
feature_names = vectorizer.get_feature_names_out()
weights = model.coef_[0]

importance = pd.DataFrame({
    "token": feature_names,
    "weight": weights
})

top_positive = importance.nlargest(20, "weight")
print("Топ-20 токенов, влияющих на Target = 1:")
print(top_positive)

plt.figure(figsize=(12, 6))
plt.bar(top_positive["token"], top_positive["weight"])
plt.xticks(rotation=45)
plt.xlabel("Токен")
plt.ylabel("Коэффициент")
plt.title("Топ-20 токенов, влияющих на Target = 1")
plt.show()

print(
    "Интерпретация: положительный коэффициент логистической регрессии повышает log-odds класса 1. "
    "Чем больше коэффициент, тем сильнее наличие токена сдвигает прогноз в сторону Target=1 "
    "при прочих равных."
)

top_negative = importance.nsmallest(20, "weight")
print("\nТоп-20 токенов, влияющих на Target = 0:")
print(top_negative)

plt.figure(figsize=(12, 6))
plt.bar(top_negative["token"], top_negative["weight"])
plt.xticks(rotation=45)
plt.xlabel("Токен")
plt.ylabel("Коэффициент")
plt.title("Топ-20 токенов, влияющих на Target = 0")
plt.show()

print(
    "Интерпретация: отрицательные коэффициенты уменьшают log-odds класса 1 и, соответственно, "
    "сдвигают решение к Target=0. Величина коэффициента отражает силу связи, но не доказывает "
    "причинность."
)

5

In [ ]:

clean_text = (
    train["Prompt_tokens_clean"].apply(lambda tokens: " ".join(tokens))
    + " "
    + train["Answer_tokens_clean"].apply(lambda tokens: " ".join(tokens))
)

x_train_clean = clean_text.loc[train_idx]
x_test_clean = clean_text.loc[test_idx]

vectorizer_clean = TfidfVectorizer()

_t0 = time.perf_counter()
x_train_clean_vec = vectorizer_clean.fit_transform(x_train_clean)
clean_vectorize_train_time = time.perf_counter() - _t0

_t0 = time.perf_counter()
x_test_clean_vec = vectorizer_clean.transform(x_test_clean)
clean_vectorize_test_time = time.perf_counter() - _t0

model_clean = LogisticRegression(max_iter=1000, class_weight="balanced")
_t0 = time.perf_counter()
model_clean.fit(x_train_clean_vec, y_train)
clean_model_train_time = time.perf_counter() - _t0

_t0 = time.perf_counter()
y_pred_clean = model_clean.predict(x_test_clean_vec)
y_score_clean = model_clean.predict_proba(x_test_clean_vec)[:, 1]
clean_model_inference_time = time.perf_counter() - _t0

clean_metrics = binary_metrics(y_test, y_pred_clean, y_score_clean)
clean_train_time = clean_vectorize_train_time + clean_model_train_time
clean_inference_time = clean_vectorize_test_time + clean_model_inference_time

clean_result = {
    "Model": "TF-IDF + LR (clean)",
    **clean_metrics,
    "Train_time_s": clean_train_time,
    "Inference_time_s": clean_inference_time,
}

comparison_step5 = pd.DataFrame([baseline_result, clean_result]).set_index("Model")
print("Шаг 5 — сравнение очищенных текстов с исходным baseline из шага 4:")
display(comparison_step5.round(4))

print("\nИзменение метрик clean - raw:")
for metric in ["Accuracy", "Precision_1", "Recall_1", "F1_1", "Balanced_accuracy", "ROC_AUC"]:
    delta = clean_metrics[metric] - baseline_metrics[metric]
    print(f"{metric}: {delta:+.4f}")

if clean_metrics["F1_1"] > baseline_metrics["F1_1"]:
    print("\nВывод: очистка улучшила F1 положительного класса относительно baseline шага 4.")
elif clean_metrics["F1_1"] < baseline_metrics["F1_1"]:
    print("\nВывод: очистка ухудшила F1 положительного класса относительно baseline шага 4.")
else:
    print("\nВывод: очистка не изменила F1 положительного класса относительно baseline шага 4.")


feature_names_clean = vectorizer_clean.get_feature_names_out()
weights_clean = model_clean.coef_[0]
importance_clean = pd.DataFrame({"token": feature_names_clean, "weight": weights_clean})
importance_clean["abs_weight"] = importance_clean["weight"].abs()
top_tokens = importance_clean.nlargest(5000, "abs_weight")["token"].tolist()

vectorizer_top = TfidfVectorizer(vocabulary=top_tokens)
_t0 = time.perf_counter()
x_train_top = vectorizer_top.fit_transform(x_train_clean)
top_vectorize_train_time = time.perf_counter() - _t0
_t0 = time.perf_counter()
x_test_top = vectorizer_top.transform(x_test_clean)
top_vectorize_test_time = time.perf_counter() - _t0

model_top = LogisticRegression(max_iter=1000, class_weight="balanced")
_t0 = time.perf_counter()
model_top.fit(x_train_top, y_train)
top_model_train_time = time.perf_counter() - _t0

_t0 = time.perf_counter()
y_pred_top = model_top.predict(x_test_top)
y_score_top = model_top.predict_proba(x_test_top)[:, 1]
top_model_inference_time = time.perf_counter() - _t0

top_metrics = binary_metrics(y_test, y_pred_top, y_score_top)
top_result = {
    "Model": "TF-IDF + LR (clean, top-5000)",
    **top_metrics,
    "Train_time_s": top_vectorize_train_time + top_model_train_time,
    "Inference_time_s": top_vectorize_test_time + top_model_inference_time,
}

print("\nДополнительное сравнение с top-5000:")
display(pd.DataFrame([baseline_result, clean_result, top_result]).set_index("Model").round(4))

6

In [ ]:
texts = (train["Prompt"] + " " + train["Answer"]).tolist()
labels = train["Target"].to_numpy()

encoded = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

x_all = encoded["input_ids"]
y_all = torch.tensor(labels, dtype=torch.float32)


x_train_cnn = x_all[train_idx.to_numpy()]
x_test_cnn = x_all[test_idx.to_numpy()]
y_train_cnn = y_all[train_idx.to_numpy()]
y_test_cnn = y_all[test_idx.to_numpy()]


class TextCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128)
        self.conv = nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = torch.relu(x)
        x = self.pool(x).squeeze(2)
        return self.fc(x).squeeze(1)  


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_cnn = TextCNN(vocab_size=tokenizer.vocab_size).to(device)


neg = (y_train_cnn == 0).sum().item()
pos = (y_train_cnn == 1).sum().item()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=0.001)

epochs = 5
train_loader = DataLoader(
    TensorDataset(x_train_cnn, y_train_cnn),
    batch_size=32,
    shuffle=True
)
test_loader = DataLoader(
    TensorDataset(x_test_cnn, y_test_cnn),
    batch_size=32
)

if device.type == "cuda":
    torch.cuda.synchronize()
_t0 = time.perf_counter()

for epoch in range(epochs):
    model_cnn.train()
    total_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        logits = model_cnn(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}: Loss = {total_loss / len(train_loader):.4f}")

if device.type == "cuda":
    torch.cuda.synchronize()
cnn_train_time = time.perf_counter() - _t0
print(f"CNN train time: {cnn_train_time:.4f} s on {device}")

In [ ]:
model_cnn.eval()
predictions = []
probabilities_all = []
true_labels = []

if device.type == "cuda":
    torch.cuda.synchronize()
_t0 = time.perf_counter()

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        logits = model_cnn(x_batch)
        probabilities = torch.sigmoid(logits)
        predicted_classes = (probabilities >= 0.5).int()

        predictions.extend(predicted_classes.cpu().numpy())
        probabilities_all.extend(probabilities.cpu().numpy())
        true_labels.extend(y_batch.numpy())

if device.type == "cuda":
    torch.cuda.synchronize()
cnn_inference_time = time.perf_counter() - _t0

cnn_metrics = binary_metrics(true_labels, predictions, probabilities_all)
cnn_result = {
    "Model": "TextCNN",
    **cnn_metrics,
    "Train_time_s": cnn_train_time,
    "Inference_time_s": cnn_inference_time,
}

print_metrics("TextCNN", cnn_metrics)
print(f"Inference time: {cnn_inference_time:.4f} s")

if cnn_metrics["Accuracy"] > 0.9 and cnn_metrics["F1_1"] < 0.3:
    print(
        "\nИнтерпретация: высокая Accuracy при низком F1_1 означает, что модель плохо распознаёт "
        "редкий класс 1. Поэтому считать CNN лучшей только по Accuracy нельзя."
    )
else:
    print(
        "\nИнтерпретация: качество CNN оцениваем по метрикам редкого класса и balanced accuracy; "
        "Accuracy при дисбалансе рассматриваем только вместе с ними."
    )

In [ ]:

results = pd.DataFrame([
    baseline_result,
    clean_result,
    top_result,
    cnn_result,
]).set_index("Model")

print("Итоговая таблица качества и скорости:")
display(results.round(4))

best_f1_model = results["F1_1"].idxmax()
fastest_train_model = results["Train_time_s"].idxmin()
fastest_inference_model = results["Inference_time_s"].idxmin()

print(f"Лучшая модель по F1 класса 1: {best_f1_model} ({results.loc[best_f1_model, 'F1_1']:.4f}).")
print(f"Самое быстрое обучение: {fastest_train_model} ({results.loc[fastest_train_model, 'Train_time_s']:.4f} s).")
print(f"Самый быстрый инференс: {fastest_inference_model} ({results.loc[fastest_inference_model, 'Inference_time_s']:.4f} s).")

print(
    "\nИтог: при несбалансированной бинарной классификации основной вывод делаем не по Accuracy, "
    "а по F1/Recall положительного класса, balanced accuracy и ROC-AUC. Время обучения и инференса "
    "показывает вычислительную цену каждого подхода и позволяет корректно сравнить TF-IDF+LR с CNN."
)